# Product & Growth Intelligence Platform
## 02 — Data Processing

### Objective

This notebook transforms the validated event-level dataset into reusable analytical structures for downstream product and growth analysis.

The processing workflow focuses on:

- preserving the validated raw event information
- creating consistent temporal features
- preparing event-level data for behavioral analysis
- building reusable analytical datasets
- integrating product-category information where required
- avoiding unnecessary duplication and memory-intensive transformations

The processed outputs from this notebook will support subsequent analysis of:

- user behavior
- funnel performance
- cohorts and retention
- behavioral segmentation
- product and category intelligence
- growth opportunities

### Processing Principle

Raw data is preserved separately from analytical datasets. Transformations are performed deliberately and only when they support a defined downstream analytical requirement.

## 1. Load Required Libraries

We begin by loading the libraries required for data processing.

At this stage, we intentionally keep the environment lightweight. Additional libraries will be introduced only when they are required by a specific processing task.

In [1]:
import pandas as pd
import numpy as np

## 2. Load Validated Event Data

The event table was validated in `01_data_understanding.ipynb`.

For this processing stage, we load the original event file and reproduce the validated cleaning rule: removal of exact duplicate records while preserving the raw source data unchanged.

In [2]:
events = pd.read_csv(r"D:\Data science portfolio\03_Product_Growth_Intelligence\data\raw\events.csv")
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")
events_clean = events.drop_duplicates().copy()
events.shape, events_clean.shape

((2756101, 6), (2755641, 6))

### Processing Validation

The raw event table is loaded without modification, and the analytical copy removes only exact duplicate records identified during the data-understanding stage.

The resulting `events_clean` table should contain the same cleaned record count established previously.

In [3]:
events_clean.duplicated().sum()

np.int64(0)

## 3. Temporal Feature Engineering

The event timestamp is converted into reusable calendar features to support time-based behavioral analysis.

These features will later support:

- daily and weekly activity trends
- funnel performance over time
- cohort construction
- retention analysis
- growth opportunity investigation

Only features with a defined downstream analytical purpose are created.

In [4]:
events_clean["date"] = events_clean["datetime"].dt.date
events_clean["year"] = events_clean["datetime"].dt.year
events_clean["month"] = events_clean["datetime"].dt.month
events_clean["week"] = events_clean["datetime"].dt.isocalendar().week
events_clean["day_of_week"] = events_clean["datetime"].dt.dayofweek
events_clean["hour"] = events_clean["datetime"].dt.hour

In [5]:
events_clean[ ["datetime", "date", "year", "month", "week", "day_of_week", "hour"]].head()

,datetime,date,year,month,week,day_of_week,hour
0,2015-06-02 05:02:12.117,2015-06-02,2015,6,23,1,5
1,2015-06-02 05:50:14.164,2015-06-02,2015,6,23,1,5
2,2015-06-02 05:13:19.827,2015-06-02,2015,6,23,1,5
3,2015-06-02 05:12:35.914,2015-06-02,2015,6,23,1,5
4,2015-06-02 05:02:17.106,2015-06-02,2015,6,23,1,5


### Temporal Processing Check

The event timestamp has been transformed into reusable calendar features while retaining the original `datetime` field.

These derived fields provide the temporal foundation for downstream trend, cohort, retention, and behavioral analysis.

## Event-Level Behavioral Indicators

To support consistent downstream behavioral analysis, the categorical `event` field is converted into binary event indicators.

Each indicator represents whether a record corresponds to a specific stage of the behavioral funnel:

- `is_view` — product view event
- `is_addtocart` — add-to-cart event
- `is_transaction` — transaction event

These indicators preserve the original `event` field while providing simple numeric measures that can be aggregated at visitor, product, session, and time levels.

The indicators will be used in subsequent funnel, engagement, conversion, and behavioral analyses.

In [6]:
events_clean["is_view"] = (events_clean["event"] == "view").astype("int8")
events_clean["is_addtocart"] = (events_clean["event"] == "addtocart").astype("int8")
events_clean["is_transaction"] = (events_clean["event"] == "transaction").astype("int8")
events_clean[ ["event", "is_view", "is_addtocart", "is_transaction"]].head()

,event,is_view,is_addtocart,is_transaction
0,view,1,0,0
1,view,1,0,0
2,view,1,0,0
3,view,1,0,0
4,view,1,0,0


In [7]:
event_indicator_totals = events_clean[
    ["is_view", "is_addtocart", "is_transaction"]
].sum()

event_indicator_totals

is_view           2664218
is_addtocart        68966
is_transaction      22457
dtype: int64

### Indicator Validation

The binary indicators correctly map each event record to its corresponding behavioral stage.

Because the original `event` column is retained, these indicators do not replace the source event classification. They provide an aggregation-friendly representation for downstream analysis.

The validated indicators will support visitor-level engagement metrics, funnel construction, session analysis, and product-level behavioral measurement.

## 4. Session Construction

Raw event records represent individual interactions, but growth analysis also requires a session-level view of user activity.

A session is defined as a sequence of events generated by the same visitor where consecutive events are separated by no more than **30 minutes of inactivity**.

A new session begins when:

- the visitor appears for the first time, or
- the time gap from the visitor's previous event exceeds 30 minutes.

The 30-minute inactivity threshold is used as an analytical sessionization rule rather than as a property explicitly provided by the source dataset.

Sessionization is performed after sorting events by `visitorid` and `datetime`. The original cleaned event table remains unchanged, while the sessionized data is stored separately for downstream analysis.

This creates a consistent session-level analytical unit for:

- engagement measurement
- funnel analysis
- session conversion
- behavioral segmentation
- retention and cohort analysis

In [8]:
# Create a sessionization-ready copy without modifying the cleaned event table
session_events = events_clean.sort_values(
    ["visitorid", "datetime"]
).copy()

# Identify the previous event timestamp for each visitor
session_events["previous_datetime"] = (
    session_events.groupby("visitorid")["datetime"].shift(1)
)

# Calculate inactivity gap in minutes
session_events["gap_minutes"] = (
    session_events["datetime"] - session_events["previous_datetime"]
).dt.total_seconds() / 60

# Start a new session for the first event or after more than 30 minutes of inactivity
session_events["new_session"] = (
    session_events["previous_datetime"].isna()
    | (session_events["gap_minutes"] > 30)
).astype("int8")

# Assign a sequential session number to each visitor
session_events["session_number"] = (
    session_events.groupby("visitorid")["new_session"].cumsum()
)

# Create a unique session identifier
session_events["session_id"] = (
    session_events["visitorid"].astype(str)
    + "_"
    + session_events["session_number"].astype(str)
)

session_events[
    [
        "visitorid",
        "datetime",
        "previous_datetime",
        "gap_minutes",
        "new_session",
        "session_number",
        "session_id"
    ]
].head(10)

,visitorid,datetime,previous_datetime,gap_minutes,new_session,session_number,session_id
1361687,0,2015-09-11 20:49:49.439,NaT,NaN,1,1,0_1
1367212,0,2015-09-11 20:52:39.591,2015-09-11 20:49:49.439,2.835867,0,1,0_1
1367342,0,2015-09-11 20:55:17.175,2015-09-11 20:52:39.591,2.626400,0,1,0_1
830385,1,2015-08-13 17:46:06.444,NaT,NaN,1,1,1_1
742616,2,2015-08-07 17:51:44.567,NaT,NaN,1,1,2_1
735273,2,2015-08-07 17:53:33.790,2015-08-07 17:51:44.567,1.820383,0,1,2_1
737711,2,2015-08-07 17:56:52.664,2015-08-07 17:53:33.790,3.314567,0,1,2_1
726292,2,2015-08-07 18:01:08.920,2015-08-07 17:56:52.664,4.270933,0,1,2_1
737615,2,2015-08-07 18:08:25.669,2015-08-07 18:01:08.920,7.279150,0,1,2_1
735202,2,2015-08-07 18:17:24.375,2015-08-07 18:08:25.669,8.978433,0,1,2_1


### Sessionization Validation

The sessionization logic is validated by checking that:

1. the first recorded event for each visitor starts a new session;
2. a new session is created only when inactivity exceeds 30 minutes;
3. events within the same session remain ordered chronologically;
4. every event receives a valid session identifier.

The validation confirms that session construction follows the predefined analytical rule before session-level metrics are calculated.

In [9]:
# Validate that every event has a session identifier
session_events["session_id"].isna().sum()

np.int64(0)

In [10]:
# Validate the session boundary rule
invalid_boundaries = session_events[
    (session_events["new_session"] == 1)
    & session_events["previous_datetime"].notna()
    & (session_events["gap_minutes"] <= 30)]
len(invalid_boundaries)

0

## 5. Session-Level Analytical Dataset

The sessionized event table provides the event-level foundation required to construct a session-level analytical dataset.

Each unique combination of `visitorid` and `session_id` represents one analytical session.

For each session, we summarize:

- session start and end time
- session duration
- total number of events
- number of unique products interacted with
- product views
- add-to-cart events
- transaction events
- number of distinct transactions
- whether the session resulted in a transaction

This transformation converts individual behavioral events into a business-oriented session view that can be used for engagement, funnel, conversion, and behavioral analysis.

In [11]:
session_summary = (
    session_events
    .groupby(["visitorid", "session_id"], as_index=False)
    .agg(
        session_start=("datetime", "min"),
        session_end=("datetime", "max"),
        event_count=("event", "size"),
        unique_products=("itemid", "nunique"),
        views=("is_view", "sum"),
        add_to_carts=("is_addtocart", "sum"),
        transaction_events=("is_transaction", "sum"),
        transactions=("transactionid", "nunique")
    )
)

session_summary["session_duration_minutes"] = (
    session_summary["session_end"] - session_summary["session_start"]
).dt.total_seconds() / 60

session_summary["has_transaction"] = (
    session_summary["transactions"] > 0
).astype("int8")

session_summary = session_summary[
    [
        "visitorid",
        "session_id",
        "session_start",
        "session_end",
        "session_duration_minutes",
        "event_count",
        "unique_products",
        "views",
        "add_to_carts",
        "transaction_events",
        "transactions",
        "has_transaction"
    ]
].sort_values(
    ["visitorid", "session_start"]
).reset_index(drop=True)

session_summary.head(10)

,visitorid,session_id,session_start,session_end,session_duration_minutes,event_count,unique_products,views,add_to_carts,transaction_events,transactions,has_transaction
0,0,0_1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,5.462267,3,3,3,0,0,0,0
1,1,1_1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,0.000000,1,1,1,0,0,0,0
2,2,2_1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,29.221300,8,4,8,0,0,0,0
3,3,3_1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,0.000000,1,1,1,0,0,0,0
4,4,4_1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,0.000000,1,1,1,0,0,0,0
5,5,5_1,2015-07-17 01:45:56.439,2015-07-17 01:45:56.439,0.000000,1,1,1,0,0,0,0
6,6,6_1,2015-08-30 06:03:48.202,2015-08-30 06:03:48.202,0.000000,1,1,0,1,0,0,0
7,6,6_2,2015-08-30 06:39:38.318,2015-08-30 06:56:33.838,16.925333,4,2,4,0,0,0,0
8,6,6_3,2015-08-31 03:21:25.697,2015-08-31 03:21:25.697,0.000000,1,1,1,0,0,0,0
9,7,7_1,2015-05-14 05:39:36.753,2015-05-14 05:42:42.817,3.101067,2,2,2,0,0,0,0


In [12]:
session_summary["session_id"].isna().sum()

np.int64(0)

In [13]:
session_summary.duplicated(
    subset=["visitorid", "session_id"]
).sum()

np.int64(0)

In [14]:
(session_summary["session_duration_minutes"] < 0).sum()

np.int64(0)

In [15]:
(session_summary["event_count"] <= 0).sum()

np.int64(0)

### Session-Level Validation Interpretation

All session-level integrity checks passed successfully.

No session records have missing session identifiers, duplicate visitor-session combinations, negative durations, or zero event counts.

The session-level dataset therefore provides a validated analytical layer between raw behavioral events and higher-level visitor, funnel, conversion, and retention analysis.

In [16]:
transaction_consistency = ( session_summary["transactions"] > session_summary["transaction_events"]).sum()
transaction_consistency

np.int64(0)

## 6. Visitor-Level Analytical Dataset

The validated session-level dataset is aggregated to the visitor level to create a behavioral profile for each visitor.

The visitor-level dataset summarizes:

- number of sessions
- active days
- total events
- total product views
- total add-to-cart events
- total transaction events
- distinct transactions
- converted sessions
- total and average session duration
- visitor-level engagement and conversion efficiency metrics
- first and last observed activity

This dataset forms the foundation for visitor engagement analysis, behavioral segmentation, conversion analysis, and retention measurement.

In [20]:
visitor_summary = (
    session_summary[
        [
            "visitorid",
            "session_id",
            "session_start",
            "session_end",
            "event_count",
            "views",
            "add_to_carts",
            "transaction_events",
            "transactions",
            "has_transaction",
            "session_duration_minutes"
        ]
    ]
    .groupby("visitorid", sort=False)
    .agg(
        total_sessions=("session_id", "nunique"),
        first_activity=("session_start", "min"),
        last_activity=("session_end", "max"),
        total_events=("event_count", "sum"),
        total_views=("views", "sum"),
        total_add_to_carts=("add_to_carts", "sum"),
        total_transaction_events=("transaction_events", "sum"),
        total_transactions=("transactions", "sum"),
        converted_sessions=("has_transaction", "sum"),
        total_session_duration_minutes=("session_duration_minutes", "sum"),
        average_session_duration_minutes=("session_duration_minutes", "mean")
    )
    .reset_index()
)

visitor_summary.head()

,visitorid,total_sessions,first_activity,last_activity,total_events,total_views,total_add_to_carts,total_transaction_events,total_transactions,converted_sessions,total_session_duration_minutes,average_session_duration_minutes
0,0,1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,3,0,0,0,0,5.462267,5.462267
1,1,1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,1,1,0,0,0,0,0.000000,0.000000
2,2,1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,8,8,0,0,0,0,29.221300,29.221300
3,3,1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,1,1,0,0,0,0,0.000000,0.000000
4,4,1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,1,1,0,0,0,0,0.000000,0.000000


In [21]:
# Calculate active days per visitor efficiently
active_days = (
    session_summary[["visitorid", "session_start"]]
    .assign(activity_date=lambda x: x["session_start"].dt.normalize())
    .drop_duplicates()
    .groupby("visitorid", sort=False)
    .size()
    .rename("active_days")
)

visitor_summary = visitor_summary.merge(
    active_days,
    on="visitorid",
    how="left"
)

visitor_summary["active_days"] = visitor_summary["active_days"].astype("int16")

visitor_summary.head()

,visitorid,total_sessions,first_activity,last_activity,total_events,total_views,total_add_to_carts,total_transaction_events,total_transactions,converted_sessions,total_session_duration_minutes,average_session_duration_minutes,active_days
0,0,1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,3,0,0,0,0,5.462267,5.462267,1
1,1,1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,1,1,0,0,0,0,0.000000,0.000000,1
2,2,1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,8,8,0,0,0,0,29.221300,29.221300,1
3,3,1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,1,1,0,0,0,0,0.000000,0.000000,1
4,4,1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,1,1,0,0,0,0,0.000000,0.000000,1


In [26]:
# Create visitor-level behavioral efficiency metrics

visitor_summary["events_per_session"] = (
    visitor_summary["total_events"]
    / visitor_summary["total_sessions"]
)

visitor_summary["views_per_session"] = (
    visitor_summary["total_views"]
    / visitor_summary["total_sessions"]
)

# Safe rate calculations
visitor_summary["cart_rate"] = np.divide(
    visitor_summary["total_add_to_carts"],
    visitor_summary["total_views"],
    out=np.zeros(len(visitor_summary), dtype=float),
    where=visitor_summary["total_views"] > 0
)

visitor_summary["transaction_rate"] = np.divide(
    visitor_summary["total_transaction_events"],
    visitor_summary["total_views"],
    out=np.zeros(len(visitor_summary), dtype=float),
    where=visitor_summary["total_views"] > 0
)

visitor_summary["session_conversion_rate"] = np.divide(
    visitor_summary["converted_sessions"],
    visitor_summary["total_sessions"],
    out=np.zeros(len(visitor_summary), dtype=float),
    where=visitor_summary["total_sessions"] > 0
)

visitor_summary.head()

,visitorid,total_sessions,first_activity,last_activity,total_events,total_views,total_add_to_carts,total_transaction_events,total_transactions,converted_sessions,total_session_duration_minutes,average_session_duration_minutes,active_days,events_per_session,views_per_session,cart_rate,transaction_rate,session_conversion_rate
0,0,1,2015-09-11 20:49:49.439,2015-09-11 20:55:17.175,3,3,0,0,0,0,5.462267,5.462267,1,3.0,3.0,0.0,0.0,0.0
1,1,1,2015-08-13 17:46:06.444,2015-08-13 17:46:06.444,1,1,0,0,0,0,0.000000,0.000000,1,1.0,1.0,0.0,0.0,0.0
2,2,1,2015-08-07 17:51:44.567,2015-08-07 18:20:57.845,8,8,0,0,0,0,29.221300,29.221300,1,8.0,8.0,0.0,0.0,0.0
3,3,1,2015-08-01 07:10:35.296,2015-08-01 07:10:35.296,1,1,0,0,0,0,0.000000,0.000000,1,1.0,1.0,0.0,0.0,0.0
4,4,1,2015-09-15 21:24:27.167,2015-09-15 21:24:27.167,1,1,0,0,0,0,0.000000,0.000000,1,1.0,1.0,0.0,0.0,0.0


In [27]:
visitor_summary.isna().sum()

visitorid                           0
total_sessions                      0
first_activity                      0
last_activity                       0
total_events                        0
total_views                         0
total_add_to_carts                  0
total_transaction_events            0
total_transactions                  0
converted_sessions                  0
total_session_duration_minutes      0
average_session_duration_minutes    0
active_days                         0
events_per_session                  0
views_per_session                   0
cart_rate                           0
transaction_rate                    0
session_conversion_rate             0
dtype: int64

In [28]:
visitor_summary[
    [
        "events_per_session",
        "views_per_session",
        "cart_rate",
        "transaction_rate",
        "session_conversion_rate"
    ]
].describe()

,events_per_session,views_per_session,cart_rate,transaction_rate,session_conversion_rate
count,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06,1.407580e+06
mean,1.422281e+00,1.387275e+00,1.351864e-02,3.499053e-03,5.448364e-03
std,1.417654e+00,1.320428e+00,1.285278e-01,5.283659e-02,6.709519e-02
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,3.100000e+02,3.000000e+02,3.100000e+01,6.000000e+00,1.000000e+00


In [29]:
(
    (visitor_summary["cart_rate"] < 0).sum(),
    (visitor_summary["transaction_rate"] < 0).sum(),
    (visitor_summary["session_conversion_rate"] < 0).sum()
)

(np.int64(0), np.int64(0), np.int64(0))

### Visitor-Level Metric Validation

The visitor-level analytical dataset contains one record per visitor and summarizes engagement, session activity, funnel interactions, and conversion behavior.

Rate-based metrics are calculated using safe division to prevent undefined values when the relevant denominator is zero. Visitors without product views therefore receive a rate of zero for view-based conversion metrics rather than an undefined value.

The resulting visitor-level dataset contains no missing analytical metrics and provides a validated foundation for downstream funnel analysis, behavioral segmentation, cohort analysis, and retention measurement.

The visitor-level dataset contains 1,407,580 unique visitors and 18 analytical features at this stage.

## 7. Product-Category Mapping

The RetailRocket dataset does not provide a direct item-to-category table.

The category hierarchy is stored separately in `category_tree.csv`, while item-level category assignments are contained within the item-property files.

Because the item-property files are large, only records where `property = "categoryid"` will be extracted. This avoids loading unnecessary product-property information into memory.

The resulting mapping will provide the product-to-category relationship required for downstream product and category intelligence.

In [30]:
category_tree = pd.read_csv(
    r"D:\Data science portfolio\03_Product_Growth_Intelligence\data\raw\category_tree.csv"
)

category_tree.head()

,categoryid,parentid
0,1016,213.0
1,809,169.0
2,570,9.0
3,1691,885.0
4,536,1691.0


In [31]:
category_tree.shape

(1669, 2)

In [32]:
category_tree.isna().sum()

categoryid     0
parentid      25
dtype: int64

In [33]:
item_properties_path_1 = (
    r"D:\Data science portfolio\03_Product_Growth_Intelligence"
    r"\data\raw\item_properties_part1.csv"
)

item_properties_path_2 = (
    r"D:\Data science portfolio\03_Product_Growth_Intelligence"
    r"\data\raw\item_properties_part2.csv"
)

category_parts = []

for file_path in [item_properties_path_1, item_properties_path_2]:
    for chunk in pd.read_csv(
        file_path,
        usecols=["timestamp", "itemid", "property", "value"],
        chunksize=250_000
    ):
        category_chunk = chunk[
            chunk["property"].eq("categoryid")
        ][["timestamp", "itemid", "value"]]

        if not category_chunk.empty:
            category_parts.append(category_chunk)

item_category_history = pd.concat(
    category_parts,
    ignore_index=True
)

item_category_history.head()

,timestamp,itemid,value
0,1435460400000,460429,1338
1,1432436400000,281245,1277
2,1435460400000,35575,1059
3,1437274800000,8313,1147
4,1437879600000,55102,47


## 7.1 Validate Item-Category Assignments

The extracted item-category history contains the category assignments recorded for individual products over time.

Before creating a final product-to-category mapping, the assignment structure is validated to determine whether products have a single category, multiple historical categories, or duplicate category records.

This prevents arbitrary category selection and ensures that the final mapping rule is based on the observed data structure.

In [34]:
# Validate the structure of item-category assignments

item_category_history["categoryid"] = pd.to_numeric(
    item_category_history["value"],
    errors="coerce"
)

assignment_summary = {
    "total_assignments": len(item_category_history),
    "unique_items": item_category_history["itemid"].nunique(),
    "unique_categories": item_category_history["categoryid"].nunique(),
    "missing_category_ids": item_category_history["categoryid"].isna().sum()
}

assignment_summary

{'total_assignments': 788214,
 'unique_items': 417053,
 'unique_categories': 1242,
 'missing_category_ids': np.int64(0)}

In [35]:
# Check whether extracted category IDs exist in the category hierarchy

valid_category_ids = set(category_tree["categoryid"])

invalid_category_assignments = item_category_history[
    ~item_category_history["categoryid"].isin(valid_category_ids)
]

len(invalid_category_assignments)

167

## 7.2 Category ID Integrity Check

The extracted item-category history contains 788,214 category assignments across 417,053 unique items.

All assignments contain a valid numeric category identifier. However, 167 assignments reference category IDs that are not present in the supplied `category_tree.csv` hierarchy.

These records are investigated before constructing the final product-category mapping rather than being silently removed. This preserves traceability and prevents unsupported assumptions about historical category assignments.

In [36]:
# Identify category assignments not present in the supplied category hierarchy

invalid_category_summary = (
    item_category_history[
        ~item_category_history["categoryid"].isin(valid_category_ids)
    ]
    .groupby("categoryid")
    .agg(
        assignment_count=("itemid", "size"),
        unique_items=("itemid", "nunique")
    )
    .sort_values("assignment_count", ascending=False)
)

invalid_category_summary

,assignment_count,unique_items
categoryid,,
306,91,61
347,10,10
919,8,8
1062,8,8
1446,7,7
1046,6,6
181,5,5
554,3,3
974,3,3


In [37]:
# Inspect the invalid category assignments

invalid_category_assignments[
    ["timestamp", "itemid", "categoryid"]
].sort_values("categoryid").head(20)

,timestamp,itemid,categoryid
495355,1431226800000,215855,80
350112,1431226800000,457073,168
241663,1431226800000,258904,171
215342,1433041200000,171412,171
602693,1431831600000,156572,181
297558,1431226800000,85044,181
51555,1435460400000,176558,181
149683,1431226800000,453514,181
739189,1431226800000,134536,181
469644,1431226800000,24937,280


In [38]:
# Count how many distinct invalid category IDs are present

invalid_category_summary.shape[0]

30

In [39]:
# Confirm the total number of affected assignments

invalid_category_summary["assignment_count"].sum()

np.int64(167)

## 7.3 Treatment of Unmatched Category Assignments

The category integrity check identified 167 item-category assignments referencing 30 category IDs that are not present in the supplied category hierarchy.

These assignments represent approximately 0.021% of all extracted item-category records and therefore have a negligible impact on the overall dataset.

Because the corresponding category definitions are unavailable in the supplied hierarchy, these unmatched assignments are excluded from the final hierarchical category mapping rather than being assigned an inferred category.

The original item-category history is retained unchanged for traceability, while only category assignments supported by the supplied hierarchy are used for downstream category-level analysis.

In [40]:
# Retain only item-category assignments supported by the supplied hierarchy

item_category_valid = item_category_history[
    item_category_history["categoryid"].isin(valid_category_ids)
].copy()

item_category_valid.shape

(788047, 4)

In [41]:
# Confirm that no unmatched category IDs remain

(
    ~item_category_valid["categoryid"].isin(valid_category_ids)
).sum()

np.int64(0)

In [42]:
# Confirm that the original history remains unchanged

len(item_category_history), len(item_category_valid)

(788214, 788047)

## 8. Final Item-Category Mapping

The validated item-category history contains category assignments recorded over time. A product may appear in multiple historical records, so the final product-level mapping must avoid arbitrary category selection.

The latest valid category assignment is used as the representative category for each item. This preserves the most recent available classification while maintaining a deterministic and reproducible mapping rule.

The historical assignment table remains available for time-aware analysis, while the resulting item-category mapping is used for downstream product-level analysis.

In [43]:
# Sort category history chronologically
item_category_valid = item_category_valid.sort_values(
    ["itemid", "timestamp"]
).copy()

# Create the final product-to-category mapping
item_category_map = (
    item_category_valid
    .drop_duplicates(subset="itemid", keep="last")
    [["itemid", "categoryid"]]
    .reset_index(drop=True)
)

item_category_map.shape

(416921, 2)

In [47]:
# Validate the final mapping

mapping_validation = {
    "unique_items": item_category_map["itemid"].nunique(),
    "unique_categories": item_category_map["categoryid"].nunique(),
    "duplicate_items": item_category_map.duplicated("itemid").sum(),
    "missing_category_ids": item_category_map["categoryid"].isna().sum()
}

mapping_validation

{'unique_items': 416921,
 'unique_categories': 1150,
 'duplicate_items': np.int64(0),
 'missing_category_ids': np.int64(0)}

In [45]:
# Confirm every mapped category exists in the supplied category hierarchy

(
    ~item_category_map["categoryid"].isin(valid_category_ids)
).sum()

np.int64(0)

## 8.1 Final Item-Category Mapping Validation

The final item-category mapping assigns one validated category to each product using the latest available category assignment.

The mapping is validated to ensure that each product has at most one representative category, no category identifiers are missing, and all assigned categories are supported by the supplied category hierarchy.

In [48]:
# Validate the final item-category mapping

mapping_validation = {
    "unique_items": item_category_map["itemid"].nunique(),
    "unique_categories": item_category_map["categoryid"].nunique(),
    "duplicate_items": item_category_map.duplicated("itemid").sum(),
    "missing_category_ids": item_category_map["categoryid"].isna().sum()
}

mapping_validation

{'unique_items': 416921,
 'unique_categories': 1150,
 'duplicate_items': np.int64(0),
 'missing_category_ids': np.int64(0)}

## 8.2 Category Hierarchy Enrichment

The validated product-to-category mapping is joined with the supplied category hierarchy to retain both the assigned category and its parent category.

This creates a hierarchical product classification that can support category-level and parent-category-level analysis without modifying the original category definitions.

In [49]:
# Attach parent-category information to each product

item_category_enriched = item_category_map.merge(
    category_tree,
    on="categoryid",
    how="left",
    validate="many_to_one"
)

item_category_enriched.head()

,itemid,categoryid,parentid
0,0,209,293.0
1,1,1114,113.0
2,2,1305,1214.0
3,3,1171,938.0
4,4,1038,1174.0


In [50]:
# Validate category hierarchy enrichment

{
    "total_products": len(item_category_enriched),
    "missing_parent_categories": item_category_enriched["parentid"].isna().sum(),
    "missing_category_matches": item_category_enriched["categoryid"].isna().sum()
}

{'total_products': 416921,
 'missing_parent_categories': np.int64(25),
 'missing_category_matches': np.int64(0)}

In [51]:
# Confirm that all mapped categories remain valid after enrichment

item_category_enriched["categoryid"].isin(
    category_tree["categoryid"]
).all()

np.True_

## 8.3 Mapping Determinism Check

The final product-category mapping uses the latest valid category assignment for each product.

Because multiple category assignments may theoretically share the same timestamp, an exact-timestamp tie check is performed to ensure that the latest-category rule produces a deterministic mapping.

If no ties are present, the latest valid assignment provides an unambiguous representative category for each product.

In [52]:
# Check for products with multiple category assignments at the same timestamp

timestamp_ties = (
    item_category_valid
    .groupby(["itemid", "timestamp"])["categoryid"]
    .nunique()
)

timestamp_ties = timestamp_ties[timestamp_ties > 1]

len(timestamp_ties)

0

## 8.4 Event-Level Product & Category Enrichment

The validated item-category mapping is joined to the cleaned event table to provide product and category context for each behavioral event.

Only the final representative category assignment for each product is used. The original event records are preserved, while category attributes are added where a valid product mapping exists.

This enriched event-level dataset forms the analytical foundation for downstream product performance, category performance, funnel, conversion, and behavioral analysis.

In [53]:
# Build the final product-to-category mapping
# using the latest valid category assignment for each product.

product_category_map = (
    item_category_valid[
        ["itemid", "timestamp", "categoryid"]
    ]
    .sort_values(["itemid", "timestamp"])
    .drop_duplicates("itemid", keep="last")
    [["itemid", "categoryid"]]
    .copy()
)

product_category_map.shape

(416921, 2)

In [54]:
# Confirm that each product has exactly one representative category

product_category_map["itemid"].duplicated().sum()

np.int64(0)

In [55]:
# Enrich the cleaned event data with the validated product-category mapping

events_enriched = events_clean.merge(
    product_category_map,
    on="itemid",
    how="left",
    validate="many_to_one"
)

events_enriched.shape

(2755641, 16)

In [56]:
# Check category enrichment coverage

enrichment_summary = {
    "total_events": len(events_enriched),
    "events_with_category": events_enriched["categoryid"].notna().sum(),
    "events_without_category": events_enriched["categoryid"].isna().sum()
}

enrichment_summary

{'total_events': 2755641,
 'events_with_category': np.int64(2500065),
 'events_without_category': np.int64(255576)}

In [57]:
# Verify that the event count was preserved by the enrichment

len(events_clean), len(events_enriched)

(2755641, 2755641)

## 8.5 Category Hierarchy Enrichment

The validated category identifiers are linked to the supplied category hierarchy using the `categoryid` and `parentid` structure.

Because the source hierarchy does not provide human-readable category names, this stage derives structural hierarchy attributes rather than inventing category labels.

The resulting hierarchy information provides category depth and root-category context for downstream product and behavioral analysis.

Events without a valid category mapping remain in the enriched event dataset so that behavioral records are not lost.

In [59]:
# Create a category lookup from the supplied hierarchy

category_lookup = (
    category_tree[
        ["categoryid", "parentid"]
    ]
    .drop_duplicates("categoryid")
    .copy()
)

category_lookup.shape

(1669, 2)

In [60]:
# Standardize category identifiers for reliable parent-child mapping

category_lookup["categoryid"] = pd.to_numeric(
    category_lookup["categoryid"],
    errors="coerce"
).astype("Int64")

category_lookup["parentid"] = pd.to_numeric(
    category_lookup["parentid"],
    errors="coerce"
).astype("Int64")

category_lookup.head()

,categoryid,parentid
0,1016,213
1,809,169
2,570,9
3,1691,885
4,536,1691


In [61]:
# Validate that every non-root parent category exists in the hierarchy

valid_category_ids = set(category_lookup["categoryid"].dropna())

invalid_parent_ids = category_lookup[
    category_lookup["parentid"].notna()
    & ~category_lookup["parentid"].isin(valid_category_ids)
]

invalid_parent_ids.shape

(0, 2)

In [62]:
# Validate category hierarchy before deriving parent-child levels

hierarchy_integrity = {
    "duplicate_category_ids": category_lookup["categoryid"].duplicated().sum(),
    "self_referencing_categories": (
        category_lookup["categoryid"]
        == category_lookup["parentid"]
    ).sum(),
    "root_categories": category_lookup["parentid"].isna().sum(),
    "total_categories": category_lookup["categoryid"].nunique()
}

hierarchy_integrity

{'duplicate_category_ids': np.int64(0),
 'self_referencing_categories': np.int64(0),
 'root_categories': np.int64(25),
 'total_categories': 1669}

In [63]:
# Validate that the category hierarchy contains no circular parent relationships

parent_map = (
    category_lookup
    .dropna(subset=["categoryid"])
    .set_index("categoryid")["parentid"]
    .to_dict()
)

def find_cycle(start_id):
    visited = set()
    current_id = start_id

    while pd.notna(current_id) and current_id in parent_map:
        if current_id in visited:
            return True

        visited.add(current_id)
        current_id = parent_map[current_id]

    return False


cycle_count = sum(
    find_cycle(category_id)
    for category_id in category_lookup["categoryid"].dropna()
)

cycle_count

0

## 8.6 Derive Category Hierarchy Levels

With the category hierarchy validated for duplicate IDs, invalid parent references, self-references, and circular relationships, hierarchy levels can be derived safely.

Each category is assigned a hierarchy depth based on its distance from a root category. This provides a consistent structural representation for downstream category and product analysis without introducing unsupported category names.

In [64]:
# Derive the hierarchy depth of each category

def get_category_depth(category_id, parent_map):
    depth = 0
    current_id = category_id
    visited = set()

    while pd.notna(current_id) and current_id in parent_map:
        if current_id in visited:
            return np.nan

        visited.add(current_id)
        parent_id = parent_map[current_id]

        if pd.isna(parent_id):
            break

        depth += 1
        current_id = parent_id

    return depth


category_lookup["category_depth"] = category_lookup["categoryid"].apply(
    lambda x: get_category_depth(x, parent_map)
)

category_lookup[
    ["categoryid", "parentid", "category_depth"]
].head()

,categoryid,parentid,category_depth
0,1016,213,3
1,809,169,3
2,570,9,3
3,1691,885,2
4,536,1691,3


In [65]:
# Validate the derived hierarchy depth

{
    "missing_depths": category_lookup["category_depth"].isna().sum(),
    "minimum_depth": category_lookup["category_depth"].min(),
    "maximum_depth": category_lookup["category_depth"].max(),
    "root_depths": (
        category_lookup.loc[
            category_lookup["parentid"].isna(),
            "category_depth"
        ].unique()
    )
}

{'missing_depths': np.int64(0),
 'minimum_depth': 0,
 'maximum_depth': 5,
 'root_depths': array([0])}

## 8.7 Final Category Reference Table

The validated category hierarchy is converted into a reusable category reference table.

The table retains the category identifier, its parent category, and its derived hierarchy depth. It serves as the authoritative category reference for downstream product and category analysis.

No category names are fabricated because the supplied hierarchy does not contain human-readable category labels.

In [66]:
# Create the final reusable category reference table

category_reference = (
    category_lookup[
        ["categoryid", "parentid", "category_depth"]
    ]
    .copy()
    .reset_index(drop=True)
)

category_reference.head()

,categoryid,parentid,category_depth
0,1016,213,3
1,809,169,3
2,570,9,3
3,1691,885,2
4,536,1691,3


In [67]:
category_reference.shape

(1669, 3)

In [68]:
category_reference.isna().sum()

categoryid         0
parentid          25
category_depth     0
dtype: int64

In [78]:
product_category_enriched = (
    product_category_map
    .merge(
        category_reference,
        on="categoryid",
        how="left",
        validate="many_to_one"
    )
)

product_category_enriched.head()

,itemid,categoryid,parentid,category_depth
0,0,209,293,2
1,1,1114,113,2
2,2,1305,1214,3
3,3,1171,938,3
4,4,1038,1174,3


In [76]:
product_category_enriched.shape

(416921, 4)

In [77]:
product_category_enriched[
    ["categoryid", "parentid", "category_depth"]
].isna().sum()

categoryid         0
parentid          25
category_depth     0
dtype: int64

## 9. Final Analytical Output Preparation

The validated processing layers created in this notebook are consolidated into a reusable analytical output layer.

The final outputs are designed to support downstream product and growth analysis without repeating expensive transformations.

The analytical layer consists of:

- cleaned event data
- session-level behavioral data
- visitor-level behavioral data
- enriched event-level product and category data
- product-category reference data
- category hierarchy reference data

These outputs preserve the validated processing decisions established throughout the notebook and provide stable inputs for downstream analysis, visualization, SQL integration, and the Product & Growth Intelligence application.

No additional analytical assumptions are introduced at this stage.

In [80]:
# Final analytical output inventory

final_outputs = {
    "events_clean": events_clean.shape,
    "session_summary": session_summary.shape,
    "visitor_summary": visitor_summary.shape,
    "events_enriched": events_enriched.shape,
    "product_category_enriched": product_category_enriched.shape,
    "category_reference": category_reference.shape
}

final_outputs

{'events_clean': (2755641, 15),
 'session_summary': (1761675, 12),
 'visitor_summary': (1407580, 18),
 'events_enriched': (2755641, 16),
 'product_category_enriched': (416921, 4),
 'category_reference': (1669, 3)}

In [82]:
# Final analytical output validation

final_validation = {
    "events_clean_rows": len(events_clean),
    "events_enriched_rows": len(events_enriched),
    "event_row_count_preserved": len(events_clean) == len(events_enriched),

    "session_summary_rows": len(session_summary),
    "visitor_summary_rows": len(visitor_summary),

    "product_category_rows": len(product_category_enriched),
    "category_reference_rows": len(category_reference)
}

final_validation

{'events_clean_rows': 2755641,
 'events_enriched_rows': 2755641,
 'event_row_count_preserved': True,
 'session_summary_rows': 1761675,
 'visitor_summary_rows': 1407580,
 'product_category_rows': 416921,
 'category_reference_rows': 1669}

## 10. Persist Analytical Outputs

The validated analytical datasets are now persisted as reusable processed outputs.

These outputs provide stable inputs for downstream SQL analysis, visualization, product and growth analysis, and application development.

The export stage uses the validated datasets created in this notebook without introducing additional transformations.

In [83]:
# Define the processed-data output directory

from pathlib import Path

processed_dir = Path(
    r"D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

processed_dir

WindowsPath('D:/Data science portfolio/03_Product_Growth_Intelligence/data/processed')

In [84]:
# Define the analytical outputs to persist

analytical_outputs = {
    "events_clean": events_clean,
    "session_summary": session_summary,
    "visitor_summary": visitor_summary,
    "events_enriched": events_enriched,
    "product_category_enriched": product_category_enriched,
    "category_reference": category_reference
}

list(analytical_outputs.keys())

['events_clean',
 'session_summary',
 'visitor_summary',
 'events_enriched',
 'product_category_enriched',
 'category_reference']

In [85]:
# Persist validated analytical outputs as CSV files

for name, df in analytical_outputs.items():
    output_path = processed_dir / f"{name}.csv"
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\events_clean.csv
Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\session_summary.csv
Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\visitor_summary.csv
Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\events_enriched.csv
Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\product_category_enriched.csv
Saved: D:\Data science portfolio\03_Product_Growth_Intelligence\data\processed\category_reference.csv


In [86]:
# Verify that all analytical output files were persisted successfully

persisted_files = {
    path.name: {
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 2)
    }
    for path in processed_dir.glob("*.csv")
}

persisted_files

{'category_reference.csv': {'exists': True, 'size_mb': 0.02},
 'events_clean.csv': {'exists': True, 'size_mb': 238.73},
 'events_enriched.csv': {'exists': True, 'size_mb': 254.1},
 'product_category_enriched.csv': {'exists': True, 'size_mb': 7.35},
 'session_summary.csv': {'exists': True, 'size_mb': 144.65},
 'visitor_summary.csv': {'exists': True, 'size_mb': 142.82}}

In [87]:
# Confirm that all required analytical outputs were persisted

expected_files = {
    "events_clean.csv",
    "session_summary.csv",
    "visitor_summary.csv",
    "events_enriched.csv",
    "product_category_enriched.csv",
    "category_reference.csv"
}

actual_files = {path.name for path in processed_dir.glob("*.csv")}

persistence_validation = {
    "expected_file_count": len(expected_files),
    "actual_file_count": len(actual_files),
    "all_required_files_present": expected_files.issubset(actual_files),
    "unexpected_files": sorted(actual_files - expected_files)
}

persistence_validation

{'expected_file_count': 6,
 'actual_file_count': 6,
 'all_required_files_present': True,
 'unexpected_files': []}

In [88]:
# Final persisted-output integrity check

persisted_shapes = {}

for name in expected_files:
    file_path = processed_dir / name
    df = pd.read_csv(file_path)
    persisted_shapes[name] = df.shape

persisted_shapes

{'category_reference.csv': (1669, 3),
 'session_summary.csv': (1761675, 12),
 'events_clean.csv': (2755641, 15),
 'events_enriched.csv': (2755641, 16),
 'visitor_summary.csv': (1407580, 18),
 'product_category_enriched.csv': (416921, 4)}